# Central configuration setup

Creates and upgrades every ETL configuration table before any pipeline notebook reads or writes configuration state.


In [ ]:
# Parameters
AUDIT_TABLE = "monitoring.cfg_silver_export_load"
TIME_PARSER_POLICY = "CORRECTED"


LOAD_FILE_CONFIG = False  # Set True for an intentional one-off bootstrap/reload.
SCHEMA_CONTRACT_CSV_PATH = "Files/cfg_files/schema_definition.csv"
DQ_RULE_CSV_PATH = "Files/cfg_files/dq_rule_definition.csv"


In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", TIME_PARSER_POLICY)


def qident(value):
    return "`" + str(value).replace("`", "``") + "`"


def qualified_name(table_name):
    schema_name, object_name = table_name.split(".", 1)
    return f"{qident(schema_name)}.{qident(object_name)}"


def ensure_delta_table(table_name, column_definitions):
    # Create a config table and add fields missing from an older deployment.
    schema_name, _ = table_name.split(".", 1)
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qident(schema_name)}")
    ddl_columns = ",\n  ".join(column_definitions)
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {qualified_name(table_name)} (\n"
        f"  {ddl_columns}\n) USING DELTA"
    )
    existing = {field.name.lower() for field in spark.table(table_name).schema.fields}
    missing = [
        definition for definition in column_definitions
        if definition.split()[0].lower() not in existing
    ]
    if missing:
        spark.sql(
            f"ALTER TABLE {qualified_name(table_name)} "
            f"ADD COLUMNS ({', '.join(missing)})"
        )
        print(f"Upgraded {table_name}: {missing}")
    return missing



In [ ]:
CONFIG_TABLE_DEFINITIONS = {
    "monitoring.cfg_silver_export_load": [
        "source_kind STRING", "source_schema STRING", "source_table STRING",
        "target_table STRING", "export_date TIMESTAMP", "status STRING",
        "reload BOOLEAN", "attempt_count INT", "run_id STRING",
        "rows_read BIGINT", "rows_written BIGINT", "duplicate_key_count BIGINT",
        "started_at TIMESTAMP", "ended_at TIMESTAMP", "error_message STRING",
        "last_updated_at TIMESTAMP",
    ],
    "monitoring.cfg_pipeline_run": [
        "run_id STRING", "pipeline_name STRING", "layer STRING",
        "source_kind STRING", "started_at TIMESTAMP", "ended_at TIMESTAMP",
        "status STRING", "tables_succeeded INT", "tables_failed INT",
        "rows_read BIGINT", "rows_written BIGINT", "error_message STRING",
    ],
    "monitoring.cfg_table_load_metric": [
        "run_id STRING", "layer STRING", "source_kind STRING",
        "source_object STRING", "target_object STRING", "rows_read BIGINT",
        "rows_written BIGINT", "duplicate_key_count BIGINT",
        "null_primary_key_count BIGINT", "recorded_at TIMESTAMP",
    ],
    "monitoring.cfg_schema_drift_definition": [
        "table_name STRING", "ordinal_position STRING", "column_name STRING",
        "data_type STRING", "is_nullable STRING", "column_default STRING",
        "primary_key_name STRING", "is_primary_key STRING",
        "foreign_key_name STRING", "referenced_schema STRING",
        "referenced_table STRING", "referenced_column STRING",
        "column_description STRING", "table_description STRING",
        "join_class STRING", "join_evidence STRING",
        "definition_hash STRING", "definition_loaded_at TIMESTAMP",
    ],
    "monitoring.cfg_schema_drift_event": [
        "run_id STRING", "source_kind STRING", "source_table STRING",
        "target_table STRING", "drift_type STRING", "column_name STRING",
        "expected_type STRING", "actual_type STRING", "referenced_table STRING",
        "referenced_column STRING", "drift_key STRING", "status STRING",
        "occurrence_count BIGINT", "first_detected_at TIMESTAMP",
        "last_detected_at TIMESTAMP", "resolved_at TIMESTAMP",
        "detected_at TIMESTAMP",
    ],
    "monitoring.cfg_month_end_gold_run": [
        "snapshot_date DATE", "status STRING", "reload BOOLEAN",
        "attempt_count INT", "run_id STRING", "started_at TIMESTAMP",
        "ended_at TIMESTAMP", "dq_result STRING", "gold_result STRING",
        "error_message STRING", "last_updated_at TIMESTAMP",
    ],
    "monitoring.cfg_archive_zip_load": [
        "zip_path STRING", "export_date TIMESTAMP", "extract_path STRING",
        "status STRING", "reload BOOLEAN", "attempt_count INT",
        "file_count INT", "run_id STRING", "started_at TIMESTAMP",
        "ended_at TIMESTAMP", "error_message STRING",
        "first_loaded_at TIMESTAMP", "last_updated_at TIMESTAMP",
    ],
    "monitoring.cfg_archive_file_load": [
        "file_path STRING", "filename STRING", "export_date TIMESTAMP",
        "source_zip STRING", "target_object STRING", "status STRING",
        "reload BOOLEAN", "attempt_count INT", "rows_read BIGINT",
        "rows_written BIGINT", "run_id STRING", "started_at TIMESTAMP",
        "ended_at TIMESTAMP", "error_message STRING",
        "first_loaded_at TIMESTAMP", "last_updated_at TIMESTAMP",
    ],
    "monitoring.cfg_archive_table_export_load": [
        "source_schema STRING", "source_table STRING", "export_date TIMESTAMP",
        "status STRING", "reload BOOLEAN", "row_count BIGINT",
        "run_id STRING", "first_seen_at TIMESTAMP",
        "last_updated_at TIMESTAMP", "error_message STRING",
    ],
    "monitoring.cfg_schema_contract_column": [
        "table_name STRING", "ordinal_position STRING", "column_name STRING",
        "data_type STRING", "is_nullable STRING", "column_default STRING",
        "primary_key_name STRING", "is_primary_key STRING",
        "foreign_key_name STRING", "referenced_schema STRING",
        "referenced_table STRING", "referenced_column STRING",
        "column_description STRING", "table_description STRING",
        "join_class STRING", "join_evidence STRING",
        "contract_loaded_at TIMESTAMP",
    ],
    "monitoring.cfg_bronze_schema_live": [
        "table_name STRING", "ordinal_position INT", "column_name STRING",
        "live_data_type STRING", "is_nullable BOOLEAN",
        "captured_at TIMESTAMP", "run_id STRING",
    ],
    "monitoring.cfg_archived_schema_live": [
        "table_name STRING", "ordinal_position STRING", "column_name STRING",
        "data_type STRING", "is_nullable STRING", "contract_loaded_at TIMESTAMP",
    ],
    "monitoring.cfg_schema_definition_candidate": [
        "table_name STRING", "ordinal_position STRING", "column_name STRING",
        "data_type STRING", "is_nullable STRING", "column_default STRING",
        "primary_key_name STRING", "is_primary_key STRING",
        "foreign_key_name STRING", "referenced_schema STRING",
        "referenced_table STRING", "referenced_column STRING",
        "column_description STRING", "table_description STRING",
    ],
    "monitoring.cfg_data_quality_result": [
        "run_id STRING", "rule_id STRING", "severity STRING",
        "rule_type STRING", "source_table STRING", "column_name STRING",
        "status STRING", "failed_row_count BIGINT", "checked_row_count BIGINT",
        "failure_percentage DOUBLE", "sample_key_json STRING",
        "checked_at TIMESTAMP", "message STRING",
    ],
    "monitoring.cfg_rejected_row": [
        "run_id STRING", "rule_id STRING", "source_table STRING",
        "business_key_json STRING", "rejection_reason STRING",
        "rejected_at TIMESTAMP",
    ],
    "monitoring.cfg_referential_exception": [
        "run_id STRING", "rule_id STRING", "child_table STRING",
        "child_column STRING", "child_key STRING", "parent_table STRING",
        "parent_column STRING", "detected_at TIMESTAMP",
    ],
    "monitoring.cfg_data_quality_rule": [
        "rule_id STRING", "active STRING", "severity STRING",
        "rule_type STRING", "source_schema STRING", "table_name STRING",
        "column_name STRING", "referenced_schema STRING",
        "referenced_table STRING", "referenced_column STRING",
        "operator STRING", "rule_value STRING", "description STRING",
        "loaded_at TIMESTAMP",
    ],
    "gold.cfg_placement_urgency_rule": [
        "PlacementUrgencyBand STRING", "MaximumTargetDays INT",
        "WarningHoursBeforeTarget INT", "SortOrder INT", "IsActive BOOLEAN",
    ],
}


upgraded_tables = {}
for config_table, definitions in CONFIG_TABLE_DEFINITIONS.items():
    missing_columns = ensure_delta_table(config_table, definitions)
    if missing_columns:
        upgraded_tables[config_table] = missing_columns

print(
    f"Configuration tables ready: {len(CONFIG_TABLE_DEFINITIONS)}; "
    f"upgraded: {len(upgraded_tables)}"
)



In [ ]:
# Stable Gold configuration belongs to setup, not the Gold model notebook.
spark.sql("""
MERGE INTO gold.cfg_placement_urgency_rule AS t
USING (
  SELECT * FROM VALUES
    ('Critical', 1, 6, 1, true), ('High', 3, 24, 2, true),
    ('Medium', 7, 48, 3, true), ('Planned', 99999, 72, 4, true),
    ('Unspecified', 99999, 72, 5, true)
  AS v(PlacementUrgencyBand, MaximumTargetDays, WarningHoursBeforeTarget, SortOrder, IsActive)
) AS s
ON t.PlacementUrgencyBand = s.PlacementUrgencyBand
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
print("Configuration setup completed")



In [ ]:
# One-off configuration bootstrap.
# Child notebooks read the Delta tables below; they never read the CSV files.
def bootstrap_csv_table(csv_path, target_table, required_columns):
    if not LOAD_FILE_CONFIG and spark.catalog.tableExists(target_table):
        # Setup creates the table shell before this cell runs. Only skip the
        # import when the existing table actually contains configuration rows.
        if spark.table(target_table).limit(1).count() > 0:
            print(f"Using existing populated {target_table}; CSV bootstrap not requested")
            return
        print(f"{target_table} exists but is empty; loading the CSV bootstrap")
    frame = (spark.read.format("csv").option("header", "true")
        .option("quote", '"').option("escape", '"').option("multiLine", "true")
        .load(csv_path))
    missing = set(required_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {sorted(missing)}")
    if frame.rdd.isEmpty():
        raise ValueError(f"{csv_path} is empty")
    (frame.withColumn("contract_loaded_at", F.current_timestamp())
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(target_table))
    print(f"Loaded {csv_path} into {target_table}: {frame.count():,} rows")

bootstrap_csv_table(
    SCHEMA_CONTRACT_CSV_PATH,
    "monitoring.cfg_schema_contract_column",
    ["table_name", "ordinal_position", "column_name", "data_type", "is_nullable",
     "is_primary_key", "referenced_schema", "referenced_table", "referenced_column",
     "join_class", "join_evidence"],
)
bootstrap_csv_table(
    DQ_RULE_CSV_PATH,
    "monitoring.cfg_data_quality_rule",
    ["rule_id", "active", "severity", "rule_type", "source_schema", "table_name",
     "column_name", "referenced_schema", "referenced_table", "referenced_column",
     "operator", "rule_value", "description"],
)
